#Reading raw data from different sources
Source data is read into the bronze layer without any tranformations

In [0]:
sourcePath = "/Volumes/data_analytics/bronzelayer/rawdata"

In [0]:
df = spark.read \
    .option("recursiveFileLookup", "true") \
    .csv(sourcePath) \
    .select("*", "_metadata.file_path")

# Writing data into tables

In [0]:

unique_paths = [row.file_path for row in df.select("file_path").distinct().collect()]
catalog_name = "data_analytics"
schema_name = "bronzelayer"

for path in unique_paths:

    clean_table_name = path.split("/")[-1].replace(".", "_").replace("-", "_")
    full_table_name = f"{catalog_name}.{schema_name}.{clean_table_name}"
    print(f"Processing: {full_table_name}")
    
    # Filter for this specific file's data and write to its own table
    df.filter(df.file_path == path) \
      .drop("file_path") \
      .write.mode("overwrite") \
      .saveAsTable(full_table_name)
